In [1]:
print("faris")

faris


In [2]:
# za svaki bat i:                     # prolazimo kroz svakog šišmiša pojedinačno
#                                     # i = indeks jednog šišmiša u populaciji
 
#     beta = random(0, 1)             # slučajan broj između 0 i 1
#                                     # koristi se da frekvencija bude malo drugačija za svakog šišmiša
 
#     f[i] = f_min + (f_max - f_min) * beta
#                                     # f[i] = frekvencija i-tog šišmiša
#                                     # frekvencija određuje koliko jako mijenja svoje kretanje
#                                     # nije "pozicija", nego parametar koji utiče na pomak
 
#     v[i] = v[i] + (x[i] - best) * f[i]
#                                     # v[i] = brzina i-tog šišmiša
#                                     # brzina govori koliko i u kojem smjeru će se pomjeriti
#                                     # x[i] = trenutna pozicija tog šišmiša
#                                     # best = trenutno najbolje rješenje od svih šišmiša
#                                     # ovom formulom šišmiš koriguje svoju brzinu u odnosu na best
 
#     x_new = x[i] + v[i]
#                                     # x_new = nova kandidatska pozicija za tog jednog šišmiša
#                                     # znači: uzmemo staru poziciju i dodamo brzinu
#                                     # DA — ovo je nova pozicija JEDNOG šišmiša, ovog i-tog
 
#     ako random(0,1) > pulse_rate[i]:
#         x_new = best + epsilon * average_loudness
#                                     # ponekad šišmiš ne ide običnim pomakom
#                                     # nego skoči blizu trenutno najboljeg rješenja
#                                     # epsilon = mali slučajni broj / slučajan mali pomak
#                                     # average_loudness = prosječna glasnoća svih šišmiša
#                                     # ovo služi za lokalnu pretragu oko najboljeg rješenja
 
#     x_new = popravi_granice(x_new)
#                                     # ako je nova pozicija izašla van dozvoljenog opsega,
#                                     # vrati je unutar granica problema
 
#     fitness_new = objective(x_new)
#                                     # izračunaj koliko je dobra nova pozicija
#                                     # objective = funkcija koju minimiziraš ili maksimiziraš
 
#     ako fitness_new < fitness[i] I random(0,1) < loudness[i]:
#         x[i] = x_new
#                                     # prihvati novu poziciju za tog šišmiša
 
#         fitness[i] = fitness_new
#                                     # zapamti novu vrijednost funkcije za tog šišmiša
 
#         loudness[i] = alpha * loudness[i]
#                                     # smanji glasnoću tog šišmiša
#                                     # što iteracije više idu, šišmiš postaje "mirniji"

In [3]:
OPCIJE = {
    # tt_split, random_state, C, kernel, gamma
    # "tt_split": [0.1, 0.2, 0.22, 0.25, 0.29, 0.3, 0.33, 0.35, 0.4, 0.45, 0.5],
    "tt_split": [0.2, 0.22, 0.25, 0.29],
    # "tt_split": np.arange(0.1, 0.5, 0.0001).tolist(),
    "random_state": [0, 1, 2, 3],
    "C": [0.1, 0.5, 1, 2, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}


# "tt_split": np.arange(0.32, 0.34, 0.0001).tolist(),

# SVM - Support Vector Machine - algoritam za klasifikaciju i regresiju

# RandomSearchCV - metoda za pronalaženje najboljih hiperparametara modela
# GridSearchCV - metoda za pronalaženje najboljih hiperparametara modela - isprobava sve kombinacije


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import random
import numpy as np

In [5]:
# Učitavanje podataka
df = pd.read_csv("iris.csv") 

X = df.drop('species', axis=1)
y = df['species']

In [6]:
X.head()

,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [7]:
y.head()

0    setosa
1    setosa
2    setosa
3    setosa
4    setosa
Name: species, dtype: object

In [8]:
# Ovaj dio koda radi evaluaciju eksperimenata dati u varijabli OPCIJE.
# Za svaki eksperiment, dijeli podatke na trening i test skup, trenira SVM model sa zadanim hiperparametrima, i računa F1 score na test skupu.
# Koristeci GridSearchCV ili RandomSearchCV bi bilo efikasnije, ali ovaj kod demonstrira osnovni pristup evaluacije.

# Najbolji parametri: 
best_params = {
    "tt_split": 0.29,
    "random_state": 0,
    "C": 0.1,
    "kernel": "linear",
    "gamma": "scale"
}

best_score = 0
best_model = None

for tt_split in OPCIJE["tt_split"]:
    for random_state in OPCIJE["random_state"]:
        for C in OPCIJE["C"]:
            for kernel in OPCIJE["kernel"]:
                for gamma in OPCIJE["gamma"]:

                    # Podjela podataka na trening i test skup
                    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=tt_split, random_state=random_state)

                    # Treniranje SVM modela
                    model = SVC(C=C, kernel=kernel, gamma=gamma)
                    model.fit(X_train, y_train)

                    # Predviđanje na test skupu
                    y_pred = model.predict(X_test)

                    # Računanje F1 score
                    score = f1_score(y_test, y_pred, average='weighted')
                    # ovdje moze ici bilo koja funkcija EVALUACIJE, npr. accuracy_score, precision_score, recall_score, itd.

                    # Ispis rezultata za trenutne parametre
                    print(f"tt_split: {tt_split}, random_state: {random_state}, C: {C}, kernel: {kernel}, gamma: {gamma} => F1 Score: {score}")
                    
                    if score > best_score:
                        best_score = score
                        best_params = {
                            "tt_split": tt_split,
                            "random_state": random_state,
                            "C": C,
                            "kernel": kernel,
                            "gamma": gamma
                        }
                        best_model = model
                    
                    
            

tt_split: 0.2, random_state: 0, C: 0.1, kernel: linear, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.1, kernel: linear, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.1, kernel: rbf, gamma: scale => F1 Score: 0.8380018674136321
tt_split: 0.2, random_state: 0, C: 0.1, kernel: rbf, gamma: auto => F1 Score: 0.9672820512820512
tt_split: 0.2, random_state: 0, C: 0.5, kernel: linear, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.5, kernel: linear, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.5, kernel: rbf, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.5, kernel: rbf, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: linear, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: linear, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: rbf, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: rbf,

In [9]:
best_params, best_score


({'tt_split': 0.2,
  'random_state': 0,
  'C': 0.1,
  'kernel': 'linear',
  'gamma': 'scale'},
 1.0)

In [10]:
best_model


,C,0.1
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [12]:
from sklearn.model_selection import RandomizedSearchCV

OPCIJE = {
    "random_state": np.arange(0, 50).tolist(),
    "C": np.arange(0.1, 10.1, 0.1).tolist(),
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"],
    "break_ties": [False],           # True samo sa decision_function_shape='ovr'
    "cache_size": [200, 300, 400],
    "class_weight": [None, 'balanced'],
    "coef0": [0.0, 0.1, 0.5, 1.0],
    "decision_function_shape": ['ovr'],  # koristimo samo ovr da se izbjegnu greške
    "degree": [3, 4, 5],
    "max_iter": [-1, 100, 200],
    "probability": [True, False],
    "shrinking": [True, False],
    "tol": [1e-3, 1e-4, 1e-5],
    "verbose": [0]
}

random_search = RandomizedSearchCV(
    estimator=SVC(),
    param_distributions=OPCIJE,
    n_iter=50,         
    scoring='f1_weighted',
    cv=5,               
    random_state=42,
    n_jobs=-1           
)

random_search.fit(X, y)

print("RandomSearchCV - Rezultati:")
print(f"Najbolji parametri: {random_search.best_params_}")
print(f"Najbolji F1 score:  {random_search.best_score_:.4f}")

RandomSearchCV - Rezultati:
Najbolji parametri: {'verbose': 0, 'tol': 0.0001, 'shrinking': False, 'random_state': 21, 'probability': False, 'max_iter': -1, 'kernel': 'rbf', 'gamma': 'scale', 'degree': 5, 'decision_function_shape': 'ovr', 'coef0': 0.1, 'class_weight': 'balanced', 'cache_size': 300, 'break_ties': False, 'C': 5.2}
Najbolji F1 score:  0.9866


In [13]:
# ALGORITAM ŠIŠMIŠA (Bat Algorithm) za optimizaciju SVM
# Parametri algoritma šišmiša
N_BATS = 10         
N_ITER = 30          
F_MIN = 0.0         
F_MAX = 2.0          
ALPHA = 0.9          
GAMMA = 0.9          
LOUDNESS_INIT = 1.0  
PULSE_RATE_INIT = 0.5  

# Diskretni parametri SVM-a koje pretražujemo
C_VALUES = [0.1, 0.5, 1, 2, 5, 10]
KERNEL_VALUES = ["linear", "rbf"]
GAMMA_VALUES = ["scale", "auto"]

N_PARAMS = 3  # [C_index, kernel_index, gamma_index]

def decode_position(pos):
    """Pretvaranje kontinuirane pozicije u diskretne SVM parametre."""
    c_idx = int(abs(pos[0])) % len(C_VALUES)
    k_idx = int(abs(pos[1])) % len(KERNEL_VALUES)
    g_idx = int(abs(pos[2])) % len(GAMMA_VALUES)
    return C_VALUES[c_idx], KERNEL_VALUES[k_idx], GAMMA_VALUES[g_idx]

def evaluate(pos, X, y, tt_split=0.2, random_state=42):
    """Evaluacija SVM model sa datim parametrima, vraća F1 score."""
    C, kernel, gamma = decode_position(pos)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=tt_split, random_state=random_state)
    model = SVC(C=C, kernel=kernel, gamma=gamma)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return f1_score(y_test, y_pred, average="weighted")

# Inicijalizacija populacije šišmiša
np.random.seed(42)

# Pozicije: svaki red je jedan šišmiš, svaki stupac jedan parametar
positions = np.random.uniform(0, len(C_VALUES), (N_BATS, N_PARAMS))
velocities = np.zeros((N_BATS, N_PARAMS))

# Inicijalizacija glasnoće i pulse rate za svakog šišmiša
loudness = np.ones(N_BATS) * LOUDNESS_INIT
pulse_rate = np.ones(N_BATS) * PULSE_RATE_INIT

# Evalucija početne pozicije
fitness = np.array([evaluate(positions[i], X, y) for i in range(N_BATS)])

# Pronalazak trenutno najboljeg rješenje
best_idx = np.argmax(fitness)
best_pos = positions[best_idx].copy()
best_score = fitness[best_idx]

print(f"Početno najbolje rješenje: {decode_position(best_pos)}, F1={best_score:.4f}")
print("="*60)

# Glavna petlja algoritma šišmiša
for iteration in range(N_ITER):
    for i in range(N_BATS):
        # 1. Generisanje nove frekvencije za šišmiša i
        beta = np.random.uniform(0, 1)
        freq = F_MIN + (F_MAX - F_MIN) * beta
        
        # 2. Ažuriranje brzine i pozicije
        velocities[i] = velocities[i] + (positions[i] - best_pos) * freq
        x_new = positions[i] + velocities[i]
        
        # 3. Lokalna pretraga: ako je random > pulse_rate, skok blizu najboljeg
        if np.random.uniform(0, 1) > pulse_rate[i]:
            average_loudness = np.mean(loudness)
            epsilon = np.random.uniform(-1, 1, N_PARAMS)
            x_new = best_pos + epsilon * average_loudness
        
        # 4. Ograničavanje pozicije na dozvoljeni raspon
        x_new = np.clip(x_new, 0, max(len(C_VALUES), len(KERNEL_VALUES), len(GAMMA_VALUES)) - 0.001)
        
        # 5. Evaluacija nove pozicije
        fitness_new = evaluate(x_new, X, y)
        
        # 6. Prihvatanje novog rješenja ako je bolje i random < loudness
        if fitness_new >= fitness[i] and np.random.uniform(0, 1) < loudness[i]:
            positions[i] = x_new
            fitness[i] = fitness_new
            
            # Ažuriranje glasnoće i pulse rate
            loudness[i] = ALPHA * loudness[i]
            pulse_rate[i] = PULSE_RATE_INIT * (1 - np.exp(-GAMMA * iteration))
        
        # 7. Ažuriranje globalno najboljeg
        if fitness[i] > best_score:
            best_score = fitness[i]
            best_pos = positions[i].copy()
    
    if (iteration + 1) % 5 == 0:
        C_best, kernel_best, gamma_best = decode_position(best_pos)
        print(f"Iteracija {iteration+1:3d}/{N_ITER}: Najboje {C_best}, {kernel_best}, {gamma_best} => F1={best_score:.4f}")

# Rezultati
C_best, kernel_best, gamma_best = decode_position(best_pos)
print("="*60)
print(f"ALGORITAM ŠIŠMIŠA - Rezultati:")
print(f"Najbolji C: {C_best}")
print(f"Najbolji kernel: {kernel_best}")
print(f"Najbolji gamma: {gamma_best}")
print(f"Najbolji F1: {best_score:.4f}")

Početno najbolje rješenje: (1, 'rbf', 'scale'), F1=1.0000
Iteracija   5/30: Najboje 1, rbf, scale => F1=1.0000
Iteracija  10/30: Najboje 1, rbf, scale => F1=1.0000
Iteracija  15/30: Najboje 1, rbf, scale => F1=1.0000
Iteracija  20/30: Najboje 1, rbf, scale => F1=1.0000
Iteracija  25/30: Najboje 1, rbf, scale => F1=1.0000
Iteracija  30/30: Najboje 1, rbf, scale => F1=1.0000
ALGORITAM ŠIŠMIŠA - Rezultati:
Najbolji C: 1
Najbolji kernel: rbf
Najbolji gamma: scale
Najbolji F1: 1.0000
